# BioDiscoveryAgent Dataset EDA

Exploratory comparison of the CRISPR screen datasets used in the paper. Run this from the `BioDiscoveryAgent` repo root so the `datasets/` and `CEGv2.txt` paths resolve correctly.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

## 2. Load all single-gene datasets

Each dataset is a `Gene, Score` table (one row per gene tested in that CRISPR screen) plus a precomputed `topmovers` file listing which genes counted as true hits. One dataset (`Steinhart_crispra_GD2_D22`) stores its columns as `0, 1` instead of `Gene, Score`, so the loader renames it for consistency.

In [ ]:
DATASET_NAMES = [
    "IFNG",
    "IL2",
    "Carnevale22_Adenosine",
    "Scharenberg22",
    "Sanchez21",
    "Sanchez21_down",
    "Steinhart_crispra_GD2_D22",
]

def load_dataset(name):
    df = pd.read_csv(f"datasets/ground_truth_{name}.csv")
    if list(df.columns) != ["Gene", "Score"]:
        df.columns = ["Gene", "Score"]
    topmovers = set(np.load(f"datasets/topmovers_{name}.npy", allow_pickle=True).tolist())
    return df, topmovers


essential = set(pd.read_csv("CEGv2.txt", delimiter="\t")["GENE"].tolist())

datasets = {}
for name in DATASET_NAMES:
    df, topmovers = load_dataset(name)
    datasets[name] = {"df": df, "topmovers": topmovers}

print(f"Loaded {len(datasets)} datasets.")

## 3. Summary table

In [ ]:
rows = []
for name, d in datasets.items():
    df, topmovers = d["df"], d["topmovers"]
    n_genes = len(df)
    n_hits = len(topmovers)
    hit_rate = n_hits / n_genes
    essential_hits = len(topmovers & essential)
    pct_essential_hits = essential_hits / n_hits if n_hits else float("nan")
    rows.append({
        "Dataset": name,
        "Genes measured": n_genes,
        "True hits": n_hits,
        "Hit rate": hit_rate,
        "Essential hits": essential_hits,
        "% of hits that are essential": pct_essential_hits,
        "Score mean": df["Score"].mean(),
        "Score std": df["Score"].std(),
    })

summary = pd.DataFrame(rows).set_index("Dataset")
summary_display = summary.copy()
summary_display["Hit rate"] = summary_display["Hit rate"].map(lambda v: f"{v:.3%}")
summary_display["% of hits that are essential"] = summary_display["% of hits that are essential"].map(lambda v: f"{v:.1%}")
summary_display["Score mean"] = summary_display["Score mean"].round(4)
summary_display["Score std"] = summary_display["Score std"].round(4)
summary_display

## 4. Score distributions across datasets

Each panel shows the histogram of raw phenotypic scores for one dataset, with a dashed line marking the minimum score among that dataset's true hits (an approximation of the hit threshold -- some datasets may have hits on both tails, see below).

In [ ]:
n = len(datasets)
ncols = 3
nrows = -(-n // ncols)  # ceil division
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for ax, (name, d) in zip(axes, datasets.items()):
    df, topmovers = d["df"], d["topmovers"]
    ax.hist(df["Score"], bins=80, color="#4C72B0", alpha=0.8)
    hit_scores = df[df["Gene"].isin(topmovers)]["Score"]
    if len(hit_scores):
        lo, hi = hit_scores.min(), hit_scores.max()
        ax.axvline(lo, color="red", linestyle="--", linewidth=1)
        ax.axvline(hi, color="red", linestyle="--", linewidth=1)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Score")
    ax.set_ylabel("Gene count")

for ax in axes[n:]:
    ax.axis("off")

fig.suptitle("Score distributions by dataset (red lines = hit score range)", y=1.02)
fig.tight_layout()
fig.savefig("score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Hit rate comparison

What fraction of genes tested in each screen ended up being a true hit? Datasets with a higher hit rate have "easier" signal -- more genes to find, which likely correlates with the higher hit ratios models achieve on them (e.g. Scharenberg22 in the paper's Table 1).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
names = list(summary.index)
hit_rates = summary["Hit rate"].values
order = np.argsort(hit_rates)[::-1]
names_sorted = [names[i] for i in order]
rates_sorted = hit_rates[order]

bars = ax.bar(names_sorted, rates_sorted, color="#55A868")
ax.set_ylabel("Hit rate (true hits / genes measured)")
ax.set_title("Hit rate by dataset")
ax.set_xticks(range(len(names_sorted)))
ax.set_xticklabels(names_sorted, rotation=30, ha="right")
for bar, v in zip(bars, rates_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.001, f"{v:.2%}",
            ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig("hit_rate_by_dataset.png", dpi=150)
plt.show()

## 6. Essential vs. non-essential composition of hits

What fraction of each dataset's true hits are "essential" genes (genes that tend to matter in almost any screen, per CEGv2.txt)? A high fraction means a model could inflate its apparent hit ratio just by guessing generically important genes -- this is exactly why the paper reports an "All genes" and a "Non-essential" hit ratio separately.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
names = list(summary.index)
essential_hits = summary["Essential hits"].values
total_hits = summary["True hits"].values
nonessential_hits = total_hits - essential_hits

order = np.argsort(total_hits)[::-1]
names_sorted = [names[i] for i in order]

ax.bar(names_sorted, nonessential_hits[order], label="Non-essential hits", color="#4C72B0")
ax.bar(names_sorted, essential_hits[order], bottom=nonessential_hits[order],
       label="Essential hits", color="#DD8452")
ax.set_ylabel("Number of true hits")
ax.set_title("Essential vs. non-essential hits by dataset")
ax.set_xticks(range(len(names_sorted)))
ax.set_xticklabels(names_sorted, rotation=30, ha="right")
ax.legend()
fig.tight_layout()
fig.savefig("essential_composition.png", dpi=150)
plt.show()

## 7. Cross-dataset hit-list overlap (Jaccard similarity)

How much do the "true hit" gene lists overlap across different screens? High similarity between two datasets suggests they're driven by overlapping biology (e.g. both immune-signaling screens); low similarity suggests mostly distinct pathways.

In [ ]:
names = list(datasets.keys())
n = len(names)
jaccard = np.zeros((n, n))

for i, j in combinations(range(n), 2):
    a = datasets[names[i]]["topmovers"]
    b = datasets[names[j]]["topmovers"]
    union = len(a | b)
    jaccard[i, j] = jaccard[j, i] = (len(a & b) / union) if union else 0.0
for i in range(n):
    jaccard[i, i] = 1.0

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(jaccard, cmap="viridis", vmin=0, vmax=jaccard[~np.eye(n, dtype=bool)].max())
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticklabels(names)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{jaccard[i, j]:.2f}", ha="center", va="center",
                color="white" if jaccard[i, j] < jaccard.max() / 2 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Jaccard similarity")
ax.set_title("Hit-list overlap across datasets")
fig.tight_layout()
fig.savefig("hit_overlap_jaccard.png", dpi=150)
plt.show()

## 8. Two-gene (combinatorial) dataset: Horlbeck

Unlike the datasets above, `Horlbeck` scores *pairs* of genes for combined (synergistic/antagonistic) effects, not single genes.

In [ ]:
horlbeck = pd.read_csv("datasets/ground_truth_Horlbeck.csv")
print(horlbeck.shape)
horlbeck.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(horlbeck["Score"], bins=100, color="#8172B2", alpha=0.85)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Synergy score")
ax.set_ylabel("Gene-pair count")
ax.set_title(f"Horlbeck: distribution of {len(horlbeck):,} gene-pair synergy scores")
fig.tight_layout()
fig.savefig("horlbeck_score_distribution.png", dpi=150)
plt.show()

print(f"Mean: {horlbeck['Score'].mean():.4f}, Std: {horlbeck['Score'].std():.4f}")
print(f"Most synergistic pair: {horlbeck.loc[horlbeck['Score'].idxmax(), 'Gene_pairs']} "
      f"({horlbeck['Score'].max():.3f})")
print(f"Most antagonistic pair: {horlbeck.loc[horlbeck['Score'].idxmin(), 'Gene_pairs']} "
      f"({horlbeck['Score'].min():.3f})")